## ChIP-Atlas Result Modification Process

- 得られた結果を、可視化用に加工しておく

In [1]:
import os
import datetime
import pathlib
import numpy as np
import pandas as pd
import polars as pl
from IPython.display import display,Image


os.environ["POLARS_MAX_THREADS"] = "16"

&nbsp;

&nbsp;

&nbsp;

## Rules for Visualization

- [ChIP-Atlas: Enrichment Analysis](https://chip-atlas.org/enrichment_analysis) で得られた結果のうち、以下に示す特徴を持つデータは取り除く

1. `Fold enrichment == 0`
2. `-log10_pvalue == -0.0`

&nbsp;

&nbsp;

## Batch Modification (per directory)

- 指定ディレクトリ直下の `*.tsv` を全て上記と同じ処理で加工し、`<input_dir>/modified/<name>_modified.tsv` として保存する

In [2]:
def modify_chip_atlas_result(tsv_path):
    """Lazily scan one ChIP-Atlas enrichment TSV and build the modification query."""
    return pl.scan_csv(
        tsv_path,
        separator="\t",
        has_header=False,
        infer_schema_length=10000,
        new_columns=[
            "experiment_id",
            "experiment_type",
            "feature",
            "cell_class",
            "cell",
            "num_of_peaks",
            "overlaps_with_cre",
            "overlaps_with_permutation_genome_region",
            "log10_pvalue",
            "log10_qvalue",
            "fold_enrichment"
        ]
    ).filter(
        pl.col("fold_enrichment") != 0
    ).with_columns(
        pl.col("fold_enrichment").log(base=2).alias("log2_fold_enrichment"),
        pl.col("log10_pvalue").neg().alias("-log10_pvalue"),
        pl.col("log10_qvalue").neg().alias("-log10_qvalue"),
        # Parse ratio columns
        (
            pl.col("overlaps_with_cre").str.split("/").list.get(0).cast(pl.Int64) /
            pl.col("overlaps_with_cre").str.split("/").list.get(1).cast(pl.Int64)
        ).alias("overlaps_with_cre_ratio"),
    ).with_columns(
        pl.col("log10_pvalue").neg().alias("-log10_pvalue"),
        pl.col("log10_qvalue").neg().alias("-log10_qvalue")
    ).drop(
        "fold_enrichment",
        "log10_pvalue",
        "log10_qvalue"
    ).filter(
        ((pl.col("-log10_pvalue") != -0.0) & # p-value が 算出できていない場合は除外
        (pl.col("-log10_qvalue") != -0.0)) # q-value が 算出できていない場合は除外
    )


def modify_chip_atlas_results_in_dir(input_dir, output_dir=None):
    """Apply modify_chip_atlas_result() to every *.tsv directly under input_dir."""
    input_dir = pathlib.Path(input_dir)
    # Default output location is <input_dir>/modified
    output_dir = pathlib.Path(output_dir) if output_dir is not None else input_dir / "modified"
    output_dir.mkdir(parents=True, exist_ok=True)

    # Only files directly under input_dir (html/, log/, modified/ are not searched)
    tsv_paths = sorted(input_dir.glob("*.tsv"))
    if not tsv_paths:
        print(f"[WARN] No TSV files found in {input_dir}")

    for tsv_path in tsv_paths:
        modified_lf = modify_chip_atlas_result(tsv_path)
        output_path = output_dir / f"{tsv_path.stem}_modified.tsv"
        # Execute the query and stream the result to file without collecting it
        modified_lf.sink_csv(output_path, separator="\t")
        print(f"{tsv_path.name} -> {output_path}")

In [3]:
chip_atlas_result_dirs = [
    "../../out/ChIP-Atlas_enrichment/up5p_all_pla_202610_v6_gncd_linked",
    "../../out/ChIP-Atlas_enrichment/up5p_all_ela_202610_v6_gncd_linked",
    "../../out/ChIP-Atlas_enrichment/down5p_all_pla_202610_v6_gncd_linked",
    "../../out/ChIP-Atlas_enrichment/down5p_all_ela_202610_v6_gncd_linked",
]

for chip_atlas_result_dir in chip_atlas_result_dirs:
    modify_chip_atlas_results_in_dir(chip_atlas_result_dir)

up5p_all_cre_pla_atac.tsv -> ../../out/ChIP-Atlas_enrichment/up5p_all_pla_202610_v6_gncd_linked/modified/up5p_all_cre_pla_atac_modified.tsv
up5p_all_cre_pla_bs.tsv -> ../../out/ChIP-Atlas_enrichment/up5p_all_pla_202610_v6_gncd_linked/modified/up5p_all_cre_pla_bs_modified.tsv
up5p_all_cre_pla_dnase.tsv -> ../../out/ChIP-Atlas_enrichment/up5p_all_pla_202610_v6_gncd_linked/modified/up5p_all_cre_pla_dnase_modified.tsv
up5p_all_cre_pla_histone.tsv -> ../../out/ChIP-Atlas_enrichment/up5p_all_pla_202610_v6_gncd_linked/modified/up5p_all_cre_pla_histone_modified.tsv
up5p_all_cre_pla_ip.tsv -> ../../out/ChIP-Atlas_enrichment/up5p_all_pla_202610_v6_gncd_linked/modified/up5p_all_cre_pla_ip_modified.tsv
up5p_all_cre_pla_pol.tsv -> ../../out/ChIP-Atlas_enrichment/up5p_all_pla_202610_v6_gncd_linked/modified/up5p_all_cre_pla_pol_modified.tsv
up5p_all_cre_pla_tf.tsv -> ../../out/ChIP-Atlas_enrichment/up5p_all_pla_202610_v6_gncd_linked/modified/up5p_all_cre_pla_tf_modified.tsv
up5p_all_cre_ela_atac.tsv 